# BUSA3022 — Part 2: Data Curation (Group Repository)

**Really Big Bank** | Sources: S&P 500 (`^GSPC`, Yahoo Finance) and VIX (`VIXCLS`, FRED).

This notebook demonstrates the **real-data proof of concept** using the 29 October 2026 validated datasets from Part 1. It makes no live API requests. All six unit tests use self-contained example inputs; do not confuse these examples with observed market anomalies.

**Inputs:** `data/validated/market_validated.csv` and `data/validated/macro_validated.csv`.
**Outputs:** `part2_data_curation/real_outputs/` — curated observations, unmatched-date audit, reject log and quality log.

## AI Agent Lens — Plain-language requirements before transformation

Before generating or changing transformation logic with an AI agent, define and verify what each metric must mean:

- **20-session moving average:** Average the latest 20 S&P 500 trading-session closing prices, including the current session. Do not calculate a value until all 20 closes are available.
- **Daily return:** Express the change from the preceding session's closing price to the current session's closing price as a percentage. The first observation has no return.
- **14-return annualised volatility:** Measure how widely the 14 most recent daily returns vary, then convert that variability to an annualised percentage with the declared 252-session convention. Do not calculate a value until 14 daily returns are available.

These are **market-session**, not calendar-day, windows. Invalid or missing observations must not be fabricated to fill the windows. The code and tests below check these requirements, including the handling of incomplete history.


## 1. Repository environment

The notebook resolves the repository root using either a local checkout or the complete repository archive when executed in Google Colab. The validated Part 1 datasets are required for the real-data curation demonstration.


In [ ]:
from pathlib import Path
import zipfile


def find_repo(places):
    for place in places:
        place = Path(place)
        if ((place / 'part2_data_curation' / 'curation.py').is_file()
                and (place / 'data' / 'validated' / 'market_validated.csv').is_file()
                and (place / 'data' / 'validated' / 'macro_validated.csv').is_file()):
            return place.resolve()
    return None

candidates = [Path.cwd(), Path.cwd().parent]
if Path('/content').exists():
    candidates += list(Path('/content').glob('BUSA3022-Analytics-Implementation*'))
    candidates += list(Path('/content').glob('**/BUSA3022-Analytics-Implementation*'))[:40]
ROOT = find_repo(candidates)
if ROOT is None:
    try:
        from google.colab import files
    except ImportError as e:
        raise FileNotFoundError('Cannot locate full repository. Start notebook from the repository root or extract the combined ZIP.') from e
    uploaded = files.upload()
    zipped = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zipped:
        raise FileNotFoundError('Upload the complete combined BUSA3022 GitHub repository ZIP, including data/validated/.')
    target = Path('/content/busa3022_uploaded')
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zipped[0]) as z:
        if any(Path(n).is_absolute() or '..' in Path(n).parts for n in z.namelist()):
            raise ValueError('Unsafe archive path: please upload a normal repository ZIP')
        z.extractall(target)
    ROOT = find_repo([target, *target.iterdir()])
if ROOT is None:
    raise FileNotFoundError('Combined repository not found: missing part2_data_curation/ or data/validated/ files.')
PART2 = ROOT/'part2_data_curation'
print('Repo:', ROOT)
print('Part 2:', PART2)

## 2. Check the code and metric formulas

The tests cover a manually checkable SMA/return calculation, 14-return volatility, duplicate and invalid-row quarantine, missing VIX values, source date and timezone treatment, audit preservation and schema failure logging.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PART2/'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=str(PART2), check=True)
print('Part 2 tests passed.')

## 3. Run curation on Part 1's actual saved validated datasets

Run the transformation only after Part 1 has produced and validated the input CSVs. S&P 500 session dates define the common calendar; missing VIX values remain missing and are flagged. A separate `vix_only_dates.csv` preserves valid macro observations without a matching S&P session.

In [ ]:
subprocess.run([
    sys.executable, str(PART2/'curation.py'),
    '--sp500', str(ROOT/'data/validated/market_validated.csv'),
    '--vix', str(ROOT/'data/validated/macro_validated.csv'),
    '--out', str(PART2/'real_outputs')
], check=True)

## 4. Output validation and source-date audit

The saved 29 October 2026 snapshot yields 124 S&P 500 sessions, 127 valid VIX dates, 123 matching dates, one missing same-day VIX record and four VIX-only dates. These summary counts are snapshot-specific.


In [ ]:
import pandas as pd
curated = pd.read_csv(PART2/'real_outputs/curated_sp500_vix.csv')
extra_vix = pd.read_csv(PART2/'real_outputs/vix_only_dates.csv')
quarantine = pd.read_csv(PART2/'real_outputs/quarantined_records.csv')
log = pd.read_csv(PART2/'real_outputs/quality_log.csv')
print('S&P sessions:', len(curated))
print('Matched dates:', int(curated['vix_close'].notna().sum()))
print('Missing same-day VIX:', int(curated['vix_gap_flag'].sum()))
print('VIX-only dates:', len(extra_vix))
print('Newly quarantined rows:', len(quarantine))
display(curated.tail(5))
display(extra_vix)
display(log.tail(5))

## 5. Independently check one daily return

The previous saved S&P 500 session (27 October 2026) closes at 7,722.6001; 28 October closes at 7,743.9800. Check the percentage return against `(today / prior_session − 1) × 100`, rather than accepting the algorithm without inspection.

In [ ]:
a = curated.loc[curated.market_session_date.eq('2026-10-28')].iloc[0]
b = curated.loc[curated.market_session_date.eq('2026-10-27')].iloc[0]
hand_return = (a.sp500_close/b.sp500_close - 1)*100
assert abs(hand_return - a.sp500_daily_return_pct) < 1e-6
print(f'Hand-checked return: {hand_return:.4f}% | Saved: {a.sp500_daily_return_pct:.4f}%')
print(f'20-session SMA: {a.sp500_sma20:.4f} | Annualised 14-return volatility: {a.sp500_volatility14_annualized_pct:.4f}%')
print('VIX missing:', bool(a.vix_gap_flag))

## 6. Small chart proof of concept (actual saved S&P 500 data)

This visualisation is only a working check of the 20-session average, **not** a completed Part 3 dashboard or a live market forecast.

In [ ]:
import matplotlib.pyplot as plt
view = curated.tail(90).copy()
view['market_session_date'] = pd.to_datetime(view['market_session_date'])
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(view['market_session_date'], view['sp500_close'], label='S&P 500 closing index')
ax.plot(view['market_session_date'], view['sp500_sma20'], label='20-session average')
ax.set(title='S&P 500 — Group validated snapshot (not live)', xlabel='US market-session date', ylabel='Index points')
ax.legend()
fig.autofmt_xdate()
plt.show()

## 7. Portfolio evidence and downstream data contract

- Metric definitions and warm-up/denominator rules are documented in the repository README.
- The Part 3 dashboard input is `part2_data_curation/real_outputs/curated_sp500_vix.csv`.
- The curation pipeline retains Part 1 source evidence and writes separate audit, reject and UTC quality logs.
- A new validated input snapshot requires regeneration of curated outputs and revalidation of published numerical results.
